In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fma_catalog.gold.mlflow_tmp;

In [0]:
import mlflow
import mlflow.spark
import pickle
import os
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import current_timestamp, col

dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")
catalog = dbutils.widgets.get("catalog_name")
silver_table = f"{catalog}.silver.audio_unsupervised"
gold_table = f"{catalog}.gold.audio_clusters"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.gold")

try:
    experiment_name = "/Shared/fma_unsupervised_clustering"
    mlflow.set_experiment(experiment_name)
    print(f"MLflow Experiment set to: {experiment_name}")
except Exception as e:
    # Fallback for restricted environments
    print("Shared path restricted, trying local workspace fallback...")
    mlflow.set_experiment("fma_clustering_local")

# Load the normalized features from Silver
silver_df = spark.table(silver_table)
print(f"Ready to train on {silver_df.count()} tracks.")

In [0]:
evaluator = ClusteringEvaluator(
    predictionCol="prediction", 
    featuresCol="scaled_features", 
    metricName="silhouette", 
    distanceMeasure="squaredEuclidean"
)

print("Testing cluster sizes (K)")

for k in range(2, 11):
    with mlflow.start_run(run_name=f"KMeans_Search_K{k}", nested=True):
        # Train Model
        kmeans = KMeans(featuresCol="scaled_features", k=k, seed=42)
        model = kmeans.fit(silver_df)
        
        # Evaluation
        predictions = model.transform(silver_df)
        silhouette_score = evaluator.evaluate(predictions)
        cost = model.summary.trainingCost
        
        # Log to MLflow
        mlflow.log_param("k", k)
        mlflow.log_metric("silhouette_score", silhouette_score)
        mlflow.log_metric("training_cost", cost)
        
        print(f"Logged K={k} | Silhouette: {silhouette_score:.4f}")

In [0]:
import mlflow.spark
import numpy as np

OPTIMAL_K = 2 
mlflow_tmp_path = f"/Volumes/{catalog}/gold/mlflow_tmp"

with mlflow.start_run(run_name="Final_Gold_Model_K2"):
    print(f"🏗️ Training final model with K={OPTIMAL_K}...")
    
    final_kmeans = KMeans(featuresCol="scaled_features", k=OPTIMAL_K, seed=42)
    final_model = final_kmeans.fit(silver_df)
    
    # 1. Log Cluster Centers
    centers = final_model.clusterCenters()
    for i, center in enumerate(centers):
        mlflow.log_dict({"center_vector": center.tolist()}, f"cluster_centers/center_{i}.json")
    
    # 2. Evaluate
    predictions_df = final_model.transform(silver_df)
    final_silhouette = evaluator.evaluate(predictions_df)
    
    # 3. Log to MLflow Registry
    mlflow.log_param("optimal_k", OPTIMAL_K)
    mlflow.log_metric("final_silhouette", final_silhouette)
    
    mlflow.spark.log_model(
        spark_model=final_model, 
        artifact_path="spark_kmeans_model",
        dfs_tmpdir=mlflow_tmp_path 
    )
    
    # --- FIXED EXPORT LOGIC FOR SERVERLESS ---
    # We save to the Volume path, not /tmp
    # Use f-string to ensure we are using the volume path
    native_save_path = f"{mlflow_tmp_path}/kmeans_model_v1"
    
    # Clean up the path if you've run this before
    dbutils.fs.rm(native_save_path, True)
    
    # Save natively to the Volume
    final_model.save(native_save_path)
    
    print(f"✅ Model registered and natively saved to Volume: {native_save_path}")

In [0]:
gold_df = (predictions_df
    .select("track_id", col("prediction").alias("cluster_id"))
    .withColumn("clustered_at", current_timestamp())
)

# Write to the Gold Delta table
(gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table))

print(f"Gold Layer Complete! {gold_df.count()} tracks successfully clustered.")